![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 01 | Framing and Baseline

*Project 3 — AI Product: From Model to Product*

**Team:**
**Track:** model / LLM
**Data:**

---

Frame the product, look at the data, fix your evaluation before you build anything, and score the baseline your solution will have to beat.

> **Done when:** your framing is written down, you know what one row or one document is and what is wrong with the data, your test set or evaluation set is fixed, and you can say your metric and your baseline score out loud.

In [ ]:
# %pip install sentence-transformers

  Obtaining dependency information for sentence-transformers from https://files.pythonhosted.org/packages/66/ae/aac4fcaf7aa27daee96aa15d74770b38bb5ff50b1d31ccf12becd68c6e8b/sentence_transformers-6.2.0-py3-none-any.whl.metadata
  Obtaining dependency information for transformers<6.0.0,>=5.0.0 from https://files.pythonhosted.org/packages/ff/bb/cc0b27a3000ddcf4903d3a26983c8b4c9a36050bbaf05675f353e8bcb08c/transformers-5.22.2-py3-none-any.whl.metadata
  Obtaining dependency information for tokenizers>=0.19 from https://files.pythonhosted.org/packages/d1/6c/24f074c9a0efb98e61b20aafe6b2641922d5db24e447d5d6daffd9e17555/tokenizers-0.23.3-cp310-abi3-win_amd64.whl.metadata
  Obtaining dependency information for huggingface-hub<3.0.0,>=1.3.0 from https://files.pythonhosted.org/packages/d4/55/ff47f1b082ecf5f43f4bc1e3b932d68eead171dabbbd1a92666c686301d7/huggingface_hub-2.4.0-py3-none-any.whl.metadata
  Obtaining dependency information for torch>=2.2 from https://files.pythonhosted.org/packages/d1/7e


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.functions import *

pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")

---
## 0. Setup

In [16]:
df = pd.read_csv("../data/raw/jobs.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (162, 7)


,id,title,company,location,description,url,search_keyword
0,5407298685,Python Developer,Global Market Solutions Germany,"Hausen, Frankfurt am Main","GMS is an operational consultancy with expertise in data and digital systems. Whether digitalisation, data & analytics or compliance - our consultants are the missing piece of our customer’s digit...",https://www.adzuna.de/details/5407298685?utm_medium=api&utm_source=87077f8e,Python
1,5931600841,Python Entwickler (m/w/d),Instaffo GmbH,"Eschborn, Main-Taunus-Kreis",Du bist interessiert an der Stelle als Python Entwickler (m/w/d) bei just.nerds GmbH & Co. KG ? Auf Instaffo kannst du dich für diesen und weitere Jobs ohne großen Aufwand bewerben. Erstelle dein ...,https://www.adzuna.de/details/5931600841?utm_medium=api&utm_source=87077f8e,Python
2,5938275963,Consultant - Python (m/w/d),x-markets consulting,"Eschborn, Main-Taunus-Kreis","Die Boutique-Beratung für den KapitalmarktSeit nunmehr 25 Jahren gestaltet x-markets consulting als kleine, feine Beratungsboutique den Kapitalmarkt mit seinen hochwertigen Leistungen für namhafte...",https://www.adzuna.de/details/5938275963?utm_medium=api&utm_source=87077f8e,Python
3,5533884834,Senior FullStack Python & JS Developer -DEU,Cryoport Systems,"Darmstadt, Hessen","Cryoport Systems is a comprehensive supply chain partner for the life sciences industry, delivering specialized solutions to meet the challenges of the biopharmaceutical, cell and gene therapy, re...",https://www.adzuna.de/details/5533884834?utm_medium=api&utm_source=87077f8e,Python
4,5940743171,Automotive Test Automation Engineer – Python & AI,Da Vinci Engineering,"Eschborn, Main-Taunus-Kreis","Stellen-ID: 22567 Zweck und Ziel der Stelle We are searching intelligent and innovative employees. If you are interested in working for a dynamic company with flat hierarchy, we look forward to re...",https://www.adzuna.de/details/5940743171?utm_medium=api&utm_source=87077f8e,Python


In [17]:
df.isnull().sum()

id                0
title             0
company           1
location          0
description       0
url               0
search_keyword    0
dtype: int64

In [18]:
print(df["description"].str.len().describe())

print("\nExample job description:\n")
print(df["description"].iloc[0])

count    162.000000
mean     497.944444
std       22.053647
min      225.000000
25%      500.000000
50%      500.000000
75%      500.000000
max      500.000000
Name: description, dtype: float64

Example job description:

GMS is an operational consultancy with expertise in data and digital systems. Whether digitalisation, data & analytics or compliance - our consultants are the missing piece of our customer’s digital puzzle. We empower their ongoing digital transformation and transform challenges into opportunities. We're part of the MoOngy Group, a more than 18 years old European network of companies with a focus on IT & Engineering expertise. Your tasks as Python developer: Bachelor's degree in Computer Scienc…


In [19]:
print("Unique job titles:", df["title"].nunique())
print("Unique companies:", df["company"].nunique())

display(df["search_keyword"].value_counts())

Unique job titles: 131
Unique companies: 97


search_keyword
Python                           20
Data Analyst                     20
Data Scientist                   19
Tableau                          19
Software engineers               19
Cloud engineers                  19
Machine Learning                 18
Backend Developer                14
Computer programmer               7
Technical support specialists     7
Name: count, dtype: int64

In [20]:
candidate_text = """
Python developer with experience in machine learning,
data analysis, SQL, Pandas, scikit-learn, and REST APIs.
Experience developing AI applications, working with
LLMs, and deploying Python applications.
Looking for a role in data science or AI engineering.
"""

In [29]:
candidate_profiles = {
    "ml_engineer": """
        Junior machine learning engineer with Python, Pandas,
        scikit-learn, SQL, data preprocessing, model evaluation,
        and experience building predictive models.
        Looking for a junior AI or machine learning position.
    """,

    "backend_developer": """
        Backend developer with Java, Spring Boot, REST APIs,
        SQL databases, unit testing, and microservices.
        Interested in backend software engineering roles.
    """,

    "data_analyst": """
        Data analyst experienced in SQL, Excel, Tableau,
        Power BI, dashboards, statistics, and reporting.
        Looking for business intelligence or data analyst roles.
    """,

    "cloud_engineer": """
        Cloud engineer with Microsoft Azure, Docker,
        Kubernetes, CI/CD, cloud infrastructure,
        and deployment automation experience.
    """,

    "technical_support": """
        IT support specialist with experience troubleshooting
        hardware and software issues, Windows administration,
        networking, help desk, and customer support.
    """
}

In [30]:
import json
from pathlib import Path

output = Path("../data/candidate_profiles.json")

with open(output, "w", encoding="utf-8") as f:
    json.dump(candidate_profiles, f, indent=4)

print(f"Saved {len(candidate_profiles)} candidate profiles")

Saved 5 candidate profiles


In [21]:
df["job_text"] = (
    df["title"].fillna("") + " " +
    df["description"].fillna("")
)

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Combine candidate profile and job descriptions
documents = [candidate_text] + df["job_text"].tolist()

# Convert text into TF-IDF vectors
vectorizer = TfidfVectorizer(stop_words="english")

tfidf_matrix = vectorizer.fit_transform(documents)

# Compare candidate against all jobs
similarities = cosine_similarity(
    tfidf_matrix[0:1],
    tfidf_matrix[1:]
).flatten()

df["tfidf_score"] = similarities

In [23]:
top_jobs = df.sort_values(
    "tfidf_score", ascending=False
).head(5)

display(
    top_jobs[
        ["title", "company", "location", "tfidf_score"]
    ]
)

,title,company,location,tfidf_score
17,Senior Cloud Developer Python | AI (m/w/d) 100.000€,SThree Germany,"Innenstadt, Aschaffenburg",0.163360
13,Cloud & AI Engineer Python (m/w/d) 100.000€,SThree Germany,"Sonnenberg, Wiesbaden",0.153385
14,Python Cloud Engineer - AI (m/w/d) 100.000€,SThree Germany,"Arheilgen, Darmstadt",0.153385
15,Cloud Software Engineer Python & AI (m/w/d) 100.000€,SThree Germany,"Altstadt, Mainz",0.150913
18,Cloud Software Engineer (m/w/d) Python & AI 100.000€,SThree Germany,"Niederrad, Frankfurt am Main",0.150913


In [24]:
feature_names = vectorizer.get_feature_names_out()

candidate_weights = tfidf_matrix[0].toarray().flatten()

word_importance = pd.DataFrame({
    "word": feature_names,
    "tfidf_weight": candidate_weights
})

display(
    word_importance
    .sort_values("tfidf_weight", ascending=False)
    .head(15)
)

,word,tfidf_weight
858,experience,0.408866
164,applications,0.378204
1887,python,0.227533
95,ai,0.212046
593,deploying,0.204433
618,developing,0.204433
1458,learn,0.204433
1487,llms,0.204433
1745,pandas,0.204433
1949,rest,0.204433


In [25]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load pretrained embedding model
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Encode candidate and job descriptions
candidate_embedding = model.encode([candidate_text])

job_embeddings = model.encode(
    df["job_text"].tolist(),
    show_progress_bar=True
)

# Calculate semantic similarity
semantic_scores = cosine_similarity(
    candidate_embedding,
    job_embeddings
).flatten()

df["semantic_score"] = semantic_scores

Batches: 100%|██████████| 6/6 [00:06<00:00,  1.05s/it]


In [26]:
comparison = pd.DataFrame({
    "TF-IDF": df.nlargest(5, "tfidf_score")["title"].tolist(),
    "Sentence Transformers": df.nlargest(5, "semantic_score")["title"].tolist()
})

display(comparison)

,TF-IDF,Sentence Transformers
0,Senior Cloud Developer Python | AI (m/w/d) 100.000€,Python Cloud Engineer - AI (m/w/d) 100.000€
1,Cloud & AI Engineer Python (m/w/d) 100.000€,Python Developer
2,Python Cloud Engineer - AI (m/w/d) 100.000€,(Senior) Software Engineer – Java & Python (m/w/d)
3,Cloud Software Engineer Python & AI (m/w/d) 100.000€,Data Analyst - Chatbot & Voicebot (w/m/d)
4,Cloud Software Engineer (m/w/d) Python & AI 100.000€,AI Cloud Platform Engineer Python (m/w/d) 100.000€


In [27]:
comparison.to_csv(
    "../data/model_comparison_1.csv",
    index=False
)

---
## 1. Frame the product

Before any code. In a few sentences:

- **The user:** who uses your product, and in what situation?
- **The input and the output:** what do they give it, and what do they get back?
- **The decision:** what do they do with the output?
- **The metric:** how will you know it works, and which mistake costs more? (*Model track:* a missed complaint or a false alarm? *LLM track:* a wrong answer, an answer with no source, or a refusal?)

You wrote a first version on launch day. Refine it here.

---
## 2. Load the data

Run `python download_data.py` (or `python download_data.py clothing`) from the repo root first. Your files land in `data/raw/`, which is gitignored.

---
## 3. First look

Shape, columns, a handful of rows **read in full**. What is one row? For the Amazon data, join a few reviews to their product in the metadata file on `parent_asin`.

---
## 4. Data quality

Empty and very short texts, HTML (`<br />`, `&#34;`), duplicates, texts in other languages, missing titles or metadata. Write down what you find and what you will do about it. Put the cleaning in a function in `src/functions.py`: the app will need it too.

---
## 5. Look at the texts

Lengths, the most frequent words, how the texts differ between ratings, products or periods. Read some hard cases: the 3-star reviews, the sarcastic ones, the ones whose rating and text disagree.

---
## 6. Model track — the target and the split

Skip this section on the LLM track.

- **Define the target** with a stated rule, for example stars to sentiment, and show the share of each class.
- **Split once**: a fixed `RANDOM_STATE`, stratified. Decide whether the split is random, by product (`parent_asin`) or by time, and say why. The test set is not touched again until notebook 03.

---
## 7. LLM track — the documents and the evaluation set

Skip this section on the model track.

- **Prepare the documents:** which texts the system will search (reviews, product descriptions, or both), how you chunk them, and the metadata each chunk keeps (product, rating, date, id), so that answers can cite them.
- **Write the evaluation set now, before you build:** at least 15 questions your user would really ask, each with the expected answer or the id of the source that answers it, and **at least 3 questions the data cannot answer**. Save it as a CSV in `data/` (it is small, and it is yours, so you can commit it).

---
## 8. The baseline

The simplest thing that could work, scored with the same evaluation as your final solution.

- **Model track:** a `DummyClassifier`, then TF-IDF with logistic regression, scored with cross-validation on the training set.
- **LLM track:** the LLM alone (no retrieval, no tools) on your evaluation set, or a keyword search (TF-IDF) for the retrieval part. Log the tokens and the cost.

---
## 9. What the baseline tells you

In prose: how good is the baseline, which mistakes does it make, and what do you expect your solution to fix?